In [1]:
import pandas as pd

In [2]:
# filter by state and county (Harris County, TX)

df = pd.read_csv("snap_retailer_location_data.csv")
tarrant_snap = df[(df["State"]== "TX") & (df["County"]=="TARRANT")]

print(tarrant_snap.shape)
print(tarrant_snap["Store_Type"].value_counts())

(1388, 17)
Store_Type
Convenience Store      685
Other                  357
Super Store            130
Grocery Store           97
Supermarket             96
Specialty Store         14
Farmers and Markets      9
Name: count, dtype: int64


In [3]:
# what is in the other cateogry
tarrant_snap[tarrant_snap["Store_Type"] == "Other"]["Store_Name"].unique()

array(['DOLLAR GENERAL 0750', 'DOLLAR GENERAL 6490', 'WALGREENS 5922',
       'Walgreens 5961', 'WALGREENS 4858', 'DOLLAR GENERAL 4295',
       'WALGREENS  3846', 'DOLLAR GENERAL 4483', 'DOLLAR GENERAL 7095',
       'Walgreens 04132', 'Walgreens 3878', 'Walgreens 06996',
       'CVS/ PHARMACY 7628', 'Lusu African Market',
       'DOLLAR GENERAL  4525', 'DOLLAR GENERAL 4982', 'Walgreens 4417',
       'DOLLAR GENERAL 3178', 'CVS/ PHARMACY 6975', 'Walgreen 4099',
       'Walgreens 04189', 'DOLLAR GENERAL 0911', 'Walgreens 03909',
       'Dollar General 0751', 'DOLLAR GENERAL 6389',
       'DOLLAR GENERAL 3631', 'DOLLAR GENERAL 7469',
       'DOLLAR GENERAL 1617', 'CVS/ PHARMACY 7678', 'DOLLAR GENERAL 1584',
       'DOLLAR GENERAL  8910', 'Walgreens 04395', 'DOLLAR GENERAL  23750',
       'DOLLAR GENERAL 3258', 'Walgreens 04291', 'DOLLAR GENERAL  3352',
       'Walgreens 03910', 'DOLLAR GENERAL 7176', 'DOLLAR GENERAL  9313',
       'Walgreens 5270', 'DOLLAR GENERAL  9203', 'WALGREENS  4857

In [4]:
# want to exclude Dollar General, Dollar Tree, CVS Pharmacy, CVS, Family Dollar,  Walgreens

excluded_others=["Dollar", "CVS", "Walgreen", "Walgrens", "Pharmacy"]

new_tarrant_data = tarrant_snap[(tarrant_snap["Store_Type"] == "Other") 
    & ~tarrant_snap["Store_Name"].str.contains("|".join(excluded_others), case=False, na=False)].copy()
print(new_tarrant_data.shape)
print(new_tarrant_data["Store_Name"].unique())

(63, 17)
['Lusu African Market' 'FARMERS MARKET OF FT WORTH' "BRAUM'S 70"
 "BRAUM'S 74" "BRAUM'S 81" "BRAUM'S 84" "BRAUM'S 124" "Braum's 128"
 "BRAUM'S 130" "BRAUM'S 165" "BRAUM'S 194" "BRAUM'S 199" "BRAUM'S 200"
 "BRAUM'S 202" "BRAUM'S 210" "BRAUM'S 215" "BRAUM'S 216" "BRAUM'S 234"
 "BRAUM'S 238" "BRAUM'S 247" "BRAUM'S 250" "Braum's 115"
 'El Tapatio Meat Market I' 'Taleeh Grocery' 'Bombay Bazaar'
 'Ocean Seafood Market' 'Guanajuato Bakery' 'Shukrani Market Inc'
 "BRAUM'S 311" 'The Joseph Storehouse' 'Lmmm Dallas 15 Ltd. 15'
 'Shukrani Market 0004' 'Anastasie African Store' 'New Diamond Grocery'
 "BRAUM'S 203" 'Demcy African Market 106' 'African Heritage Market'
 "Mire's Market" "BRAUM'S 132" 'African Union Market'
 'Blessing African Store 1' 'Shadai Group African Market 1'
 'Leelah African Store' 'Afro Mart' 'Dgs Exotic Market'
 'La Michoacana Meat Market D22' 'Spice And Slice Bazaar'
 "Lizzy's African Market" 'Nath Beauty Collection And African Store'
 'Gloesis Group' 'Charme Market

In [5]:
# I also excluded any place that said closed on google maps
kept_tarrant_others = ["Lusu African Market", "FARMERS MARKET OF FT WORTH", "El Tapatio Meat Market I", "Bombay Bazaar", "Ocean Seafood Market",
                     "The Joseph Storehouse", "Lmmm Dallas 15 Ltd. 15", "Anastasie African Store", "New Diamond Grocery", "African Heritage Market", "African Union Market"
                     , "Blessing African Store", "Shadai Group African Market","La Michoacana Meat Market", "Spice And Slice Bazaar", "Lizzy's African Market",
                     "Charme Market", "Akabanga International Market", "Namaste Grocery", "El Mercadito Supermarket Inc", "Favor African Market Llc", "La Michoacana Meat Market D22" ]

excluded_tarrant_others = ["Taleeh Grocery", "Guanajuato Bakery", "Shukrani Market Inc'", "Demcy African Market", "Mire's Market", "Leelah African Store"
                          ,"Afro Mart", "Dgs Exotic Marke","Nath Beauty Collection And African Store", "Gloesis Group","A&E International Market",
                         "Lifeland Market Llc", "Md Divine Supermarket", "Jm Farm Market", "Afriq All Collection", "African Pride Food Store" ]

In [6]:
# filter out stores that are not grocery stores
included = ["Super Store","Supermarket","Grocery Store", "Farmers and Markets"]

cleaned_snap_data = tarrant_snap[tarrant_snap["Store_Type"].isin(included) | 
    tarrant_snap["Store_Name"].isin(kept_tarrant_others)| 
    tarrant_snap["Store_Name"].str.contains("Braum", case=False, na=False)].copy()

In [7]:
# remove columns that are not needed :"Additonal_Address", "Incentive_Program", "Grantee_Name", "X", "Y"

final_cleaned_snap_data = cleaned_snap_data.drop(columns=["Additonal_Address", "Incentive_Program", "Grantee_Name", "X", "Y", "Zip4"])

In [8]:
# Check for duplicate store names at the same address
dupes = final_cleaned_snap_data[cleaned_snap_data.duplicated(subset=["Store_Name", "Store_Street_Address"], keep=False)]
print(dupes[["Store_Name", "Store_Street_Address"]])

Empty DataFrame
Columns: [Store_Name, Store_Street_Address]
Index: []


In [9]:
# final csv file created
final_cleaned_snap_data.to_csv("cleaned_tarrant_snap_data.csv", index=False)
print(final_cleaned_snap_data.shape)
print(final_cleaned_snap_data[["Store_Name", "Latitude", "Longitude", "Store_Type"]].head())

(375, 11)
                                Store_Name   Latitude  Longitude  \
1681  Iyanu African Caribbean Marketplace   32.768894 -97.098106   
1683     Sana Restaurant And Halal Grocery  32.861015 -97.252441   
2702           Victory Tropical Market Inc  32.842609 -97.082672   
3541              Whole Foods Market 10098  32.766853 -97.099075   
4464                     Target Store 1765  32.889248 -97.256554   

         Store_Type  
1681  Grocery Store  
1683  Grocery Store  
2702  Grocery Store  
3541    Super Store  
4464    Super Store  
